In [ ]:
"""
============================================================================
 Fine-grained MI EEG — ORIGINAL pipeline + MINIMAL determinism patches
 Runs ONE joint pair across all time windows, saves per-subject results.

 USAGE (Colab):
   1. RESTART RUNTIME  (CUBLAS_WORKSPACE_CONFIG must be set before CUDA init)
   2. Mount Drive
   3. Edit CLASS_A / CLASS_B below.  Nothing else.
   4. Run top to bottom.  Saves to /content/drive/MyDrive/mi_results/<PAIR>/
   5. Repeat for all 28 pairs.

 ---------------------------------------------------------------------------
 WHAT CHANGED FROM YOUR ORIGINAL  (everything else is verbatim)
 ---------------------------------------------------------------------------
 [P1] CUBLAS_WORKSPACE_CONFIG set BEFORE `import torch`.
      Set after CUDA init it silently does nothing.
 [P2] torch.use_deterministic_algorithms(True, warn_only=False)
      warn_only=True warned, then used the nondeterministic kernel anyway.
      THIS WAS THE ROOT CAUSE.
 [P3] EfficientChannelAttention: AdaptiveMaxPool1d -> x.amax(...)
      Mathematically identical; AdaptiveMaxPool1d has a nondeterministic
      CUDA backward and now raises under P2.
 [P4] DataLoaders get worker_init_fn=seed_worker.
 [P5] Seeds cast to python int  (np.int64 breaks random.seed()).
 [P6] Stale hardcoded "Class 0 vs Class 6" print now reads CLASS_A/CLASS_B.
 [P7] Per-pair saving of per-subject + per-fold accuracies.

 EXPLICITLY *NOT* CHANGED (I broke these last time — reverted):
   - model selection stays  `if val_acc > best_val_acc`  (your original).
     Once P1-P4 remove the underlying FP nondeterminism, the tie-breaking
     is reproducible, so val-loss selection was never needed.
   - classifier head keeps  Linear(emb,128)+ReLU+Dropout(0.4)+Linear(128,n)
   - FeatureFusionModule / DepthwiseSeparableConv1d / MultiscaleTemporalBlock
     / SpatialFeatureExtraction unchanged
   - augmentation, normalization, CV structure, epochs, LR unchanged
============================================================================
"""
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    pass  # not running in Google Colab
# ===========================================================================
# [P1] DETERMINISM BOOTSTRAP — must run on a FRESH runtime, before torch/CUDA
# ===========================================================================
import os

seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # [P1] before CUDA init

import random
import glob
import re
import json
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# [P2] was warn_only=True -> silently fell back to nondeterministic kernels
torch.use_deterministic_algorithms(True, warn_only=False)

generator = torch.Generator()
generator.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device} | gpu={GPU} | torch={torch.__version__}')
print('NOTE: determinism holds per-GPU-model. Record this GPU name.')


# [P4] deterministic DataLoader workers
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# ===========================================================================
# CONFIG  — the ONLY thing you edit between pairs
# ===========================================================================
CLASS_A = 2
CLASS_B = 6                    # <<< EDIT ME

JOINT_NAMES = {0: "HOC", 1: "WFE", 2: "WAA", 3: "EPS",
               4: "EFE", 5: "SPS", 6: "SAA", 7: "SFE"}
PAIR_NAME = f"{JOINT_NAMES[CLASS_A]}/{JOINT_NAMES[CLASS_B]}"   # [P6]
PAIR_SLUG = PAIR_NAME.replace("/", "_")

DATASET_ROOT = '/content/drive/MyDrive/multi_joint_mi_dataset/extracted/FineMI/FineMI'
RESULTS_ROOT = '/content/drive/MyDrive/mi_results'
SAMPLING_RATE = 250

TEST_TIME_WINDOWS_MS = [800, 1500, 3000, 4000]
TEST_TIME_WINDOWS_SAMPLES = [int(t * SAMPLING_RATE / 1000) for t in TEST_TIME_WINDOWS_MS]

print(f"\nPAIR: {PAIR_NAME}  (classes {CLASS_A} vs {CLASS_B})")
print(f"Test time windows: {TEST_TIME_WINDOWS_MS} ms")
print(f"Test time windows (samples): {TEST_TIME_WINDOWS_SAMPLES}")


# ===========================================================================
# HELPERS  (verbatim from your notebook)
# ===========================================================================
def init_weights(m):
    """Initialize model weights deterministically."""
    if isinstance(m, (nn.Conv1d, nn.Linear)):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm1d):
        torch.nn.init.ones_(m.weight)
        torch.nn.init.zeros_(m.bias)


def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        eeg = batch['eeg'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(eeg)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            eeg = batch['eeg'].to(device)
            labels = batch['label'].to(device)

            outputs = model(eeg)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc, all_preds, all_labels


def z_score_normalize(X_train, X_val, X_test):
    """Per-channel z-score normalization using training data statistics only."""
    channel_means = np.mean(X_train, axis=(0, 2))
    channel_stds = np.std(X_train, axis=(0, 2))
    channel_stds = np.where(channel_stds == 0, 1.0, channel_stds)

    channel_means = channel_means[np.newaxis, :, np.newaxis]
    channel_stds = channel_stds[np.newaxis, :, np.newaxis]

    X_train_norm = (X_train - channel_means) / channel_stds
    X_val_norm = (X_val - channel_means) / channel_stds
    X_test_norm = (X_test - channel_means) / channel_stds

    return X_train_norm, X_val_norm, X_test_norm


def add_gaussian_noise_augmentation(X, noise_level=0.1, random_seed=None):
    """Add Gaussian noise to EEG data for augmentation.  DA = DR + NL*GN"""
    if random_seed is not None:
        np.random.seed(int(random_seed))            # [P5] int cast

    channel_stds = np.std(X, axis=(0, 2), keepdims=True)
    noise = np.random.normal(loc=0.0, scale=noise_level * channel_stds, size=X.shape)
    return X + noise


NOISE_LEVEL = 0.15


# ===========================================================================
# MODEL  (verbatim, except [P3] inside EfficientChannelAttention.forward)
# ===========================================================================
class EfficientChannelAttention(nn.Module):
    """Efficient Channel Attention block."""

    def __init__(self, channels, reduction=4):
        super(EfficientChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # self.max_pool = nn.AdaptiveMaxPool1d(1)   # [P3] removed: nondeterministic backward

        self.fc = nn.Sequential(
            nn.Conv1d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv1d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch, channels, time)
        avg_out = self.fc(self.avg_pool(x))
        # [P3] x.amax(dim=-1, keepdim=True) == AdaptiveMaxPool1d(1)(x), deterministic
        max_out = self.fc(x.amax(dim=-1, keepdim=True))
        out = avg_out + max_out
        return self.sigmoid(out) * x


class DepthwiseSeparableConv1d(nn.Module):
    """Depthwise Separable Convolution block."""

    def __init__(self, in_channels, out_channels, kernel_size, groups, stride=1, padding=0):
        super(DepthwiseSeparableConv1d, self).__init__()
        self.depthwise = nn.Conv1d(
            in_channels, in_channels, kernel_size,
            stride=stride, padding=padding, groups=groups, bias=False
        )
        self.bn1 = nn.BatchNorm1d(in_channels)

        self.pointwise = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.pointwise(x)
        x = self.bn2(x)
        x = self.relu(x)

        return x


class FeatureFusionModule(nn.Module):
    """Feature Fusion Module with Depthwise Separable Conv and ECA."""

    def __init__(self, channels, kernel_size=(1, 5), groups=None):
        super(FeatureFusionModule, self).__init__()
        if groups is None:
            groups = channels

        k_size = kernel_size[1] if isinstance(kernel_size, tuple) else kernel_size
        self.ds_conv = DepthwiseSeparableConv1d(
            channels, channels, k_size, groups=groups, padding=k_size // 2
        )
        self.eca = EfficientChannelAttention(channels)

    def forward(self, x):
        x = self.ds_conv(x)
        x = self.eca(x)
        return x


class MultiscaleTemporalBlock(nn.Module):
    """Multiscale temporal convolution with 3 parallel branches."""

    def __init__(self, in_channels, out_channels_per_branch=32):
        super(MultiscaleTemporalBlock, self).__init__()

        self.branch1 = nn.Sequential(
            nn.Conv1d(in_channels, out_channels_per_branch, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(out_channels_per_branch),
            nn.ReLU()
        )
        self.branch2 = nn.Sequential(
            nn.Conv1d(in_channels, out_channels_per_branch, kernel_size=15, padding=7, bias=False),
            nn.BatchNorm1d(out_channels_per_branch),
            nn.ReLU()
        )
        self.branch3 = nn.Sequential(
            nn.Conv1d(in_channels, out_channels_per_branch, kernel_size=31, padding=15, bias=False),
            nn.BatchNorm1d(out_channels_per_branch),
            nn.ReLU()
        )

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        return torch.cat([out1, out2, out3], dim=1)


class SpatialFeatureExtraction(nn.Module):
    """Spatial feature extraction with Conv and MaxPool."""

    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(SpatialFeatureExtraction, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size,
                      padding=kernel_size // 2, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU()
        )
        self.maxpool = nn.MaxPool1d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.maxpool(x)
        return x


class CNNEarlyClassificationModel(nn.Module):
    """CNN model for early EEG classification."""

    def __init__(self, n_channels, n_timepoints, n_classes=2, embedding_dim=128):
        super(CNNEarlyClassificationModel, self).__init__()

        self.multiscale_temporal = MultiscaleTemporalBlock(
            in_channels=n_channels, out_channels_per_branch=32)
        temporal_out_channels = 32 * 3

        self.spatial_extraction = SpatialFeatureExtraction(
            in_channels=temporal_out_channels, out_channels=64)

        self.feature_fusion = FeatureFusionModule(
            channels=64, kernel_size=(1, 5), groups=64)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.cnn_projection = nn.Sequential(
            nn.Linear(64, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # NOTE: two-layer head — this is your original. Do not simplify.
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes)
        )

    def forward(self, x_eeg):
        x = self.multiscale_temporal(x_eeg)
        x = self.spatial_extraction(x)
        x = self.feature_fusion(x)
        x = self.global_pool(x)
        x = x.squeeze(-1)
        cnn_embedding = self.cnn_projection(x)
        return self.classifier(cnn_embedding)


class EEGDataset(Dataset):
    """Dataset for EEG data."""

    def __init__(self, eeg_data, labels):
        self.eeg_data = torch.FloatTensor(eeg_data)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {'eeg': self.eeg_data[idx], 'label': self.labels[idx]}


# ===========================================================================
# LOAD DATA + BINARY FILTER + TIME WINDOWS
# ===========================================================================
def get_subject_number(path):
    basename = os.path.basename(path)
    match = re.search(r"subject(\d+)", basename, re.IGNORECASE)
    return int(match.group(1)) if match else 0


all_data, all_labels, subject_ids_list = [], [], []
subject_folders = sorted(
    [f for f in glob.glob(os.path.join(DATASET_ROOT, "subject*")) if os.path.isdir(f)],
    key=get_subject_number)
print(f"\nFound {len(subject_folders)} subject folders")

for subject_path in subject_folders:
    sid = get_subject_number(subject_path)
    eeg_candidates = glob.glob(os.path.join(DATASET_ROOT, f"subject{sid}_eeg_epochs_*.npz"))
    if not eeg_candidates:
        print(f"  Skipping subject{sid}: EEG .npz not found")
        continue
    z = np.load(eeg_candidates[0], allow_pickle=True)
    all_data.append(z["data"])
    all_labels.append(z["labels"])
    subject_ids_list.extend([sid] * len(z["labels"]))

X_full = np.concatenate(all_data, axis=0)
y_full = np.concatenate(all_labels, axis=0)
subject_ids = np.array(subject_ids_list)

print(f"\nBINARY CLASSIFICATION: Class {CLASS_A} vs Class {CLASS_B}  ({PAIR_NAME})")
binary_mask = (y_full == CLASS_A) | (y_full == CLASS_B)
X_full = X_full[binary_mask]
y_full = y_full[binary_mask]
subject_ids = subject_ids[binary_mask]
y_full = np.where(y_full == CLASS_B, 1, 0)
print(f"  Class 0 (original {CLASS_A}): {np.sum(y_full == 0)} trials")
print(f"  Class 1 (original {CLASS_B}): {np.sum(y_full == 1)} trials")

full_len = X_full.shape[2]
TEST_TIME_WINDOWS_SAMPLES_FINAL = [min(s, full_len) for s in TEST_TIME_WINDOWS_SAMPLES]
TEST_TIME_WINDOWS_MS_FINAL = list(TEST_TIME_WINDOWS_MS)

time_windows_data = {}
print("\nExtracted time windows:")
for t_ms, t_samp in zip(TEST_TIME_WINDOWS_MS_FINAL, TEST_TIME_WINDOWS_SAMPLES_FINAL):
    time_windows_data[t_ms] = X_full[:, :, :t_samp]
    print(f"  {t_ms}ms ({t_samp} samples): shape {time_windows_data[t_ms].shape}")

n_channels = X_full.shape[1]
unique_subjects = [int(s) for s in np.unique(subject_ids)]     # [P5] python ints

subject_data = {}
for subj_id in unique_subjects:
    subj_mask = (subject_ids == subj_id)
    subject_data[subj_id] = {'X_full': X_full[subj_mask],
                             'y': y_full[subj_mask],
                             'n_trials': int(np.sum(subj_mask))}
    for t_ms in TEST_TIME_WINDOWS_MS_FINAL:
        subject_data[subj_id][f'X_{t_ms}ms'] = time_windows_data[t_ms][subj_mask]

batch_size = 32
n_epochs = 50
n_classes = 2

print(f"\nModel configuration:")
print(f"  Channels: {n_channels}")
print(f"  Time windows: {TEST_TIME_WINDOWS_MS_FINAL} ms")
# [P6] was a hardcoded "Class 0 vs Class 6" string
print(f"  Classes: {n_classes} (Binary: Class {CLASS_A} vs Class {CLASS_B})")
print(f"  Batch size: {batch_size}")
print(f"  Epochs per subject per time window: {n_epochs}")


# ===========================================================================
# WITHIN-SUBJECT TRAINING + TESTING  (structure verbatim)
# ===========================================================================
all_subject_results = []

for subj_idx, subj_id in enumerate(unique_subjects):
    subj_id = int(subj_id)                                    # [P5]
    X_subj_time_windows = {t: subject_data[subj_id][f'X_{t}ms']
                           for t in TEST_TIME_WINDOWS_MS_FINAL}
    y_subj = subject_data[subj_id]['y']

    if len(np.unique(y_subj)) < 2 or len(y_subj) < 10:
        print(f"Skipping Subject {subj_id}: insufficient data")
        continue

    n_folds = 5
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True,
                          random_state=int(42 + subj_id))      # [P5]
    subj_indices = np.arange(len(y_subj))
    # same splits reused across every window -> windows are truly paired
    cv_splits = list(skf.split(subj_indices, y_subj))

    subject_time_window_results = {}

    for tw_idx, t_ms in enumerate(TEST_TIME_WINDOWS_MS_FINAL):
        t_samples = TEST_TIME_WINDOWS_SAMPLES_FINAL[tw_idx]
        X_subj_tw = X_subj_time_windows[t_ms]
        fold_results_tw = []

        for fold_idx, (train_val_indices, test_indices) in enumerate(cv_splits):
            train_indices, val_indices = train_test_split(
                train_val_indices, test_size=0.25,
                random_state=int(42 + subj_id),                # [P5]
                stratify=y_subj[train_val_indices])

            X_train, y_train = X_subj_tw[train_indices], y_subj[train_indices]
            X_val, y_val = X_subj_tw[val_indices], y_subj[val_indices]
            X_test, y_test = X_subj_tw[test_indices], y_subj[test_indices]

            X_train_norm, X_val_norm, X_test_norm = z_score_normalize(X_train, X_val, X_test)

            aug_seed = int(seed + int(subj_id) + int(fold_idx) + int(tw_idx) * 100)
            X_train_augmented = add_gaussian_noise_augmentation(
                X_train_norm, noise_level=NOISE_LEVEL, random_seed=aug_seed)
            X_train_final = np.concatenate([X_train_norm, X_train_augmented], axis=0)
            y_train_final = np.concatenate([y_train, y_train], axis=0)

            train_dataset = EEGDataset(X_train_final, y_train_final)
            val_dataset = EEGDataset(X_val_norm, y_val)
            test_dataset = EEGDataset(X_test_norm, y_test)

            loader_seed = int(seed + int(subj_id) * 1000 + int(fold_idx) * 100 + int(tw_idx))
            train_generator = torch.Generator()
            train_generator.manual_seed(loader_seed)

            # [P4] worker_init_fn added to all three loaders
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                      num_workers=0, generator=train_generator,
                                      worker_init_fn=seed_worker)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                                    num_workers=0, worker_init_fn=seed_worker)
            test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                                     num_workers=0, worker_init_fn=seed_worker)

            model_init_seed = int(seed + int(subj_id) * 1000 + int(fold_idx) * 100 + int(tw_idx))
            torch.manual_seed(model_init_seed)                 # [P5] int cast
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(model_init_seed)

            model = CNNEarlyClassificationModel(
                n_channels=n_channels, n_timepoints=t_samples,
                n_classes=n_classes, embedding_dim=128).to(device)
            model.apply(init_weights)

            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=5)

            best_val_acc = 0.0
            best_model_state = None

            for epoch in range(n_epochs):
                train_loss, train_acc = train_epoch(model, train_loader, criterion,
                                                    optimizer, device)
                val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
                scheduler.step(val_loss)

                # ORIGINAL selection rule. Deterministic once P1-P4 are in place.
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_model_state = model.state_dict().copy()

            if best_model_state is not None:
                model.load_state_dict(best_model_state)

            test_loss, test_acc, test_preds, test_labels = validate(
                model, test_loader, criterion, device)
            test_preds = np.array(test_preds)
            test_labels = np.array(test_labels)

            fold_results_tw.append({
                'fold': fold_idx + 1,
                'test_accuracy': test_acc,
                'test_loss': test_loss,
                'test_preds': test_preds,
                'test_labels': test_labels,
                'n_test_samples': len(test_labels),
                'val_accuracy': best_val_acc,
            })

        all_test_preds_tw = np.concatenate([r['test_preds'] for r in fold_results_tw])
        all_test_labels_tw = np.concatenate([r['test_labels'] for r in fold_results_tw])

        subject_time_window_results[t_ms] = {
            'test_accuracy': accuracy_score(all_test_labels_tw, all_test_preds_tw) * 100,
            'test_preds': all_test_preds_tw,
            'test_labels': all_test_labels_tw,
            'confusion_matrix': confusion_matrix(all_test_labels_tw, all_test_preds_tw),
            'n_test_samples': len(all_test_labels_tw),
            'fold_results': fold_results_tw,
        }

    all_subject_results.append({'subject_id': subj_id, 'n_folds': n_folds,
                                'time_window_results': subject_time_window_results})

    print(f"S{subj_id:>2}: " + "  ".join(
        f"{t}ms={subject_time_window_results[t]['test_accuracy']:6.2f}%"
        for t in TEST_TIME_WINDOWS_MS_FINAL))

print(f"\nALL SUBJECTS COMPLETED — {PAIR_NAME}")


# ===========================================================================
# [P7] SAVE PER-SUBJECT + PER-FOLD RESULTS.  Nothing needs to run after this.
# ===========================================================================
PAIR_DIR = os.path.join(RESULTS_ROOT, PAIR_SLUG)
os.makedirs(PAIR_DIR, exist_ok=True)

subj_rows, fold_rows = [], []
for r in sorted(all_subject_results, key=lambda x: x['subject_id']):
    sid = int(r['subject_id'])
    for w in TEST_TIME_WINDOWS_MS_FINAL:
        twr = r['time_window_results'][w]
        subj_rows.append(dict(pair=PAIR_NAME, subject=sid, window_ms=w,
                              accuracy=float(twr['test_accuracy']),
                              n_test=int(twr['n_test_samples'])))
        for f in twr['fold_results']:
            fold_rows.append(dict(pair=PAIR_NAME, subject=sid, window_ms=w,
                                  fold=int(f['fold']),
                                  accuracy=float(f['test_accuracy']),
                                  val_accuracy=float(f['val_accuracy']),
                                  n_test=int(f['n_test_samples'])))

df_subj = pd.DataFrame(subj_rows)
df_fold = pd.DataFrame(fold_rows)

# subjects x windows, fixed row/column order — this is what the paired tests consume
wide = (df_subj.pivot(index='subject', columns='window_ms', values='accuracy')
        .reindex(columns=TEST_TIME_WINDOWS_MS_FINAL).sort_index())

df_subj.to_csv(f"{PAIR_DIR}/per_subject.csv", index=False)
df_fold.to_csv(f"{PAIR_DIR}/per_fold.csv", index=False)
wide.to_csv(f"{PAIR_DIR}/wide_subject_x_window.csv")

payload = dict(pair=PAIR_NAME, class_a=int(CLASS_A), class_b=int(CLASS_B),
               windows=TEST_TIME_WINDOWS_MS_FINAL, seed=seed,
               gpu=GPU, torch=torch.__version__,
               subjects=[int(s) for s in wide.index],
               matrix=wide.to_numpy().tolist())
with open(f"{PAIR_DIR}/results.pkl", "wb") as f:
    pickle.dump(payload, f)
with open(f"{PAIR_DIR}/results.json", "w") as f:
    json.dump(payload, f, indent=2)

print(f"\n{'=' * 68}")
print(f"SAVED {PAIR_NAME} -> {PAIR_DIR}")
print(f"gpu={GPU} | torch={torch.__version__} | seed={seed}")
print(f"{'=' * 68}")
print(wide.round(2).to_string())
print("\nmean:", wide.mean().round(2).to_dict())
print("sd  :", wide.std(ddof=1).round(2).to_dict())
if wide.isna().any().any():
    print("WARNING: missing subject/window cells — a subject was skipped.")

n_saved = len([d for d in os.listdir(RESULTS_ROOT)
               if os.path.isdir(os.path.join(RESULTS_ROOT, d)) and not d.startswith('_')])
print(f"\nPairs saved so far: {n_saved}/28")


# ===========================================================================
# REPRODUCIBILITY CHECK
#   Run ONE pair twice, restarting the runtime in between, changing nothing.
#   Copy the first output dir to <PAIR>_run2, then:
#
#     a = pd.read_csv(f"{RESULTS_ROOT}/HOC_SAA/wide_subject_x_window.csv", index_col=0)
#     b = pd.read_csv(f"{RESULTS_ROOT}/HOC_SAA_run2/wide_subject_x_window.csv", index_col=0)
#     print("max |diff| =", (a - b).abs().values.max(), "pp")
#
#   0.0        -> deterministic; proceed with all 28 pairs
#   non-zero   -> check the printed gpu= name matched across both runs
# ===========================================================================

Mounted at /content/drive
Using device: cuda | gpu=NVIDIA A100-SXM4-40GB | torch=2.11.0+cu128
NOTE: determinism holds per-GPU-model. Record this GPU name.

PAIR: WAA/SAA  (classes 2 vs 6)
Test time windows: [800, 1500, 3000, 4000] ms
Test time windows (samples): [200, 375, 750, 1000]

Found 18 subject folders


Exception ignored in: <function NpzFile.__del__ at 0x7b796332a660>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 226, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 221, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected



BINARY CLASSIFICATION: Class 2 vs Class 6  (WAA/SAA)
  Class 0 (original 2): 725 trials
  Class 1 (original 6): 725 trials

Extracted time windows:
  800ms (200 samples): shape (1450, 62, 200)
  1500ms (375 samples): shape (1450, 62, 375)
  3000ms (750 samples): shape (1450, 62, 750)
  4000ms (1000 samples): shape (1450, 62, 1000)

Model configuration:
  Channels: 62
  Time windows: [800, 1500, 3000, 4000] ms
  Classes: 2 (Binary: Class 2 vs Class 6)
  Batch size: 32
  Epochs per subject per time window: 50
S 1: 800ms= 54.44%  1500ms= 65.56%  3000ms= 63.33%  4000ms= 61.11%
S 2: 800ms= 55.00%  1500ms= 43.75%  3000ms= 51.25%  4000ms= 60.00%
S 3: 800ms= 68.75%  1500ms= 81.25%  3000ms= 77.50%  4000ms= 86.25%
S 4: 800ms= 76.25%  1500ms= 68.75%  3000ms= 87.50%  4000ms= 86.25%
S 5: 800ms= 75.00%  1500ms= 62.50%  3000ms= 45.00%  4000ms= 46.25%
S 6: 800ms= 46.25%  1500ms= 76.25%  3000ms= 60.00%  4000ms= 66.25%
S 7: 800ms= 78.75%  1500ms= 81.25%  3000ms= 80.00%  4000ms= 80.00%
S 8: 800ms= 47.50

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    pass  # not running in Google Colab

Mounted at /content/drive


In [ ]:
# REPRODUCIBILITY CHECK
#   Run ONE pair twice, restarting the runtime in between, changing nothing.
#   Copy the first output dir to <PAIR>_run2, then:
#
a = pd.read_csv(f"{RESULTS_ROOT}/HOC_SAA/wide_subject_x_window.csv", index_col=0)
b = pd.read_csv(f"{RESULTS_ROOT}/HOC_SAA_run2/wide_subject_x_window.csv", index_col=0)
print("max |diff| =", (a - b).abs().values.max(), "pp")
#
#   0.0        -> deterministic; proceed with all 28 pairs
#   non-zero   -> check the printed gpu= name matched across both runs

max |diff| = 0.0 pp
